# 1. Setup & file paths

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Dataset paths
DATA_DIR = Path("../data/raw/m5")

CALENDAR_PATH = DATA_DIR / "calendar.csv"
SALES_PATH = DATA_DIR / "sales_train_validation.csv"
PRICE_PATH = DATA_DIR / "sell_prices.csv"

# Verify files
print("Calendar exists:", CALENDAR_PATH.exists())
print("Sales exists:", SALES_PATH.exists())
print("Prices exists:", PRICE_PATH.exists())

print("\nData Cleaning Environment Ready!")

Calendar exists: True
Sales exists: True
Prices exists: True

Data Cleaning Environment Ready!


# 2. Checking missing values

In [3]:
# 1. Calendar missing values
calendar = pd.read_csv(CALENDAR_PATH)

print("CALENDAR MISSING VALUES")
print(calendar.isnull().sum())

# 2. Sales dataset missing values (chunk-wise)
sales_missing = pd.Series(dtype="int64")

for chunk in pd.read_csv(SALES_PATH, chunksize=5000):
    missing = chunk.isnull().sum()
    sales_missing = sales_missing.add(missing, fill_value=0)

print("\nSALES MISSING VALUES")
print(sales_missing.astype(int))

# 3. Pricing dataset missing values (chunk-wise)
price_missing = pd.Series(dtype="int64")

for chunk in pd.read_csv(PRICE_PATH, chunksize=100000):
    missing = chunk.isnull().sum()
    price_missing = price_missing.add(missing, fill_value=0)

print("\nPRICING MISSING VALUES")
print(price_missing.astype(int))

CALENDAR MISSING VALUES
date               0
wm_yr_wk           0
weekday            0
wday               0
month              0
year               0
d                  0
event_name_1    1807
event_type_1    1807
event_name_2    1964
event_type_2    1964
snap_CA            0
snap_TX            0
snap_WI            0
dtype: int64

SALES MISSING VALUES
cat_id      0
d_1         0
d_10        0
d_100       0
d_1000      0
           ..
dept_id     0
id          0
item_id     0
state_id    0
store_id    0
Length: 1919, dtype: int64

PRICING MISSING VALUES
item_id       0
sell_price    0
store_id      0
wm_yr_wk      0
dtype: int64


*We have used the chunks here in the above cell so that we don't need to load whole dataset/ file once at a time in the memory!
##### *Observation: # Missing values in calendar:
            -- event__name_1 & event_type_1:- 1807 missing.
            -- event_name_2 & event_type_2:- 1964 missing.
            *these are not necessarily errors. Not each & everyday has to be a special day/event in a calendar.

# 3. Handle calendar missing values

Expected: The missing values should be 0 in all 4 columns.

In [4]:
# Fill missing event information
event_columns = [
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2"
]

calendar[event_columns] = calendar[event_columns].fillna("No Event")

# Verify missing values
print("Missing values after cleaning:")
print(calendar[event_columns].isnull().sum())

print("\nUpdated Event Records:")
display(calendar[event_columns].head(10))

Missing values after cleaning:
event_name_1    0
event_type_1    0
event_name_2    0
event_type_2    0
dtype: int64

Updated Event Records:


,event_name_1,event_type_1,event_name_2,event_type_2
0,No Event,No Event,No Event,No Event
1,No Event,No Event,No Event,No Event
2,No Event,No Event,No Event,No Event
3,No Event,No Event,No Event,No Event
4,No Event,No Event,No Event,No Event
5,No Event,No Event,No Event,No Event
6,No Event,No Event,No Event,No Event
7,No Event,No Event,No Event,No Event
8,SuperBowl,Sporting,No Event,No Event
9,No Event,No Event,No Event,No Event


# 4. Duplicate Records Analysis:

In [5]:
# 1. Calendar duplicate check
calendar_duplicates = calendar.duplicated(subset=["d"]).sum()

# 2. Sales duplicate check
sales_ids = pd.read_csv(SALES_PATH, usecols=["id"])
sales_duplicates = sales_ids.duplicated(subset=["id"]).sum()

# 3. Pricing duplicate check
price_keys = pd.read_csv(
    PRICE_PATH,
    usecols=["store_id", "item_id", "wm_yr_wk"]
)

price_duplicates = price_keys.duplicated().sum()

# Display results
print("DUPLICATE RECORDS ANALYSIS")
print("--------------------------------")
print("Calendar duplicates:", calendar_duplicates)
print("Sales duplicates:", sales_duplicates)
print("Pricing duplicates:", price_duplicates)

DUPLICATE RECORDS ANALYSIS
--------------------------------
Calendar duplicates: 0
Sales duplicates: 0
Pricing duplicates: 0


# 5. Data Type Validation

Checking wether the data type of the columns is correct or not

In [6]:
# Load small samples for data type inspection
sales_sample = pd.read_csv(SALES_PATH, nrows=5)
price_sample = pd.read_csv(PRICE_PATH, nrows=5)

print("CALENDAR DATA TYPES")
print(calendar.dtypes)

print("\nSALES DATA TYPES")
print(sales_sample.dtypes)

print("\nPRICING DATA TYPES")
print(price_sample.dtypes)

CALENDAR DATA TYPES
date              str
wm_yr_wk        int64
weekday           str
wday            int64
month           int64
year            int64
d                 str
event_name_1      str
event_type_1      str
event_name_2      str
event_type_2      str
snap_CA         int64
snap_TX         int64
snap_WI         int64
dtype: object

SALES DATA TYPES
id            str
item_id       str
dept_id       str
cat_id        str
store_id      str
            ...  
d_1909      int64
d_1910      int64
d_1911      int64
d_1912      int64
d_1913      int64
Length: 1919, dtype: object

PRICING DATA TYPES
store_id          str
item_id           str
wm_yr_wk        int64
sell_price    float64
dtype: object


Observation: Only in calendar, the date format has to be changed from 'str' to 'int'.

# 6. Data type conversion(Dates in calendar)

In [6]:
# Convert date column to datetime
calendar["date"] = pd.to_datetime(calendar["date"])

# Verify conversion
print("Updated Calendar Data Types:")
print(calendar[["date", "wm_yr_wk"]].dtypes)

print("\nDate Range:")
print(calendar["date"].min())
print(calendar["date"].max())

Updated Calendar Data Types:
date        datetime64[us]
wm_yr_wk             int64
dtype: object

Date Range:
2011-01-29 00:00:00
2016-06-19 00:00:00


why we converted the date format?
it will be required in further analyses to sort, filter, grouping & plotting.

# 7. Verifying the updated data type

In [7]:
print("FINAL DATA TYPE CHECK")

print("\nCalendar:")
print(calendar.dtypes)

print("\nPricing:")
print(price_sample.dtypes)

daily_columns = [
    col for col in sales_sample.columns
    if col.startswith("d_")
]

print("\nSales numerical columns:")
print(sales_sample[daily_columns[:5]].dtypes)

FINAL DATA TYPE CHECK

Calendar:
date              str
wm_yr_wk        int64
weekday           str
wday            int64
month           int64
year            int64
d                 str
event_name_1      str
event_type_1      str
event_name_2      str
event_type_2      str
snap_CA         int64
snap_TX         int64
snap_WI         int64
dtype: object

Pricing:
store_id          str
item_id           str
wm_yr_wk        int64
sell_price    float64
dtype: object

Sales numerical columns:
d_1    int64
d_2    int64
d_3    int64
d_4    int64
d_5    int64
dtype: object


# 8. Sales data validation

checking whether any -ve value is thier or not in sales quantities & the overall distribution of daily sales.(using chunking)

In [8]:
negative_sales = 0
total_sales = 0
zero_sales = 0

for chunk in pd.read_csv(SALES_PATH, chunksize=5000):
    daily_data = chunk.filter(regex=r"^d_\d+$")

    negative_sales += (daily_data < 0).sum().sum()
    zero_sales += (daily_data == 0).sum().sum()
    total_sales += daily_data.size

print("Total Sales Values:", total_sales)
print("Negative Sales Values:", negative_sales)
print("Zero Sales Values:", zero_sales)

Total Sales Values: 58327370
Negative Sales Values: 0
Zero Sales Values: 39777094


#### Observation: Around 68.2%sales are zero.
This is expected in M5 data. No product could be sold daily.
and this is called "Intermittent demand"

# 9. Sales Data Summary

checking overall sales distribution

In [9]:
total_units_sold = 0
max_daily_sales = 0
min_positive_sales = float("inf")

for chunk in pd.read_csv(SALES_PATH, chunksize=5000):
    daily_data = chunk.filter(regex=r"^d_\d+$")

    total_units_sold += daily_data.sum().sum()

    max_daily_sales = max(
        max_daily_sales,
        daily_data.max().max()
    )

    positive_values = daily_data[daily_data > 0]

    if not positive_values.empty:
        min_positive_sales = min(
            min_positive_sales,
            positive_values.min().min()
        )

print("Total Units Sold:", total_units_sold)
print("Maximum Daily Sales:", max_daily_sales)
print("Minimum Positive Sales:", min_positive_sales)

Total Units Sold: 65695409
Maximum Daily Sales: 763
Minimum Positive Sales: 1.0


Observation: 65.69 million of total units is the combined sales volume of all 30,490 products-store records and 1,913 days.
WHERE, 
Maximum daily sales763 means the highest recorded sale of oneday, was 763 units of any product-store combination.

# 9. Pricing Data Validation

Checking if their is any -ve or nun value in the pricing dataset, & What is the actual price range.

In [10]:
min_price = float("inf")
max_price = float("-inf")
non_positive_prices = 0
total_price_records = 0

for chunk in pd.read_csv(PRICE_PATH, chunksize=100000):

    prices = chunk["sell_price"]

    min_price = min(min_price, prices.min())
    max_price = max(max_price, prices.max())

    non_positive_prices += (prices <= 0).sum()
    total_price_records += len(prices)

print("Total Price Records:", total_price_records)
print("Minimum Price:", min_price)
print("Maximum Price:", max_price)
print("Zero/Negative Prices:", non_positive_prices)

Total Price Records: 6841121
Minimum Price: 0.01
Maximum Price: 107.32
Zero/Negative Prices: 0


Observation: No invalid or non-positive price valuefound

# 10. Saving the cleaned calendar data

In [11]:
# Create processed data directory
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Save cleaned calendar
calendar.to_csv(
    PROCESSED_DIR / "calendar_cleaned.csv",
    index=False
)

print("Cleaned calendar saved successfully!")
print("File:", PROCESSED_DIR / "calendar_cleaned.csv")

Cleaned calendar saved successfully!
File: ../data/processed/calendar_cleaned.csv


# 11. Final Verification of File

In [12]:
# Verify processed file
cleaned_calendar_path = PROCESSED_DIR / "calendar_cleaned.csv"

print("File Exists:", cleaned_calendar_path.exists())

if cleaned_calendar_path.exists():
    print("File Size:", round(cleaned_calendar_path.stat().st_size / 1024, 2), "KB")

    cleaned_calendar = pd.read_csv(cleaned_calendar_path)
    print("Shape:", cleaned_calendar.shape)
    print("Missing Values:", cleaned_calendar.isnull().sum().sum())

File Exists: True
File Size: 159.97 KB
Shape: (1969, 14)
Missing Values: 0
